### Imports

In [ ]:
import sys
import os

sys.path.append(os.path.abspath("..")) 

import json
import pandas as pd
from datetime import datetime
from typing import Optional
from tqdm.auto import tqdm
from clean_text import clean_text
from format_date import format_date
from generate_doc_id import generate_doc_id

### Helper functions for cleaning

In [ ]:
def parse_date(date_str: str) -> str:
    """Parse date string handling multiple formats and languages"""
    if not date_str:
        raise Exception("No date str found")
    
    # Spanish to English month mapping
    spanish_months = {
        'enero': 'January', 'febrero': 'February', 'marzo': 'March',
        'abril': 'April', 'mayo': 'May', 'junio': 'June',
        'julio': 'July', 'agosto': 'August', 'septiembre': 'September',
        'octubre': 'October', 'noviembre': 'November', 'diciembre': 'December'
    }
    
    # Replace Spanish month names with English
    date_lower = date_str.lower()
    for spanish, english in spanish_months.items():
        if spanish in date_lower:
            date_str = date_str.replace(spanish.capitalize(), english)
            date_str = date_str.replace(spanish, english)
            break
    
    # Try to parse the date
    date_formats = ['%B %d, %Y', '%b %d, %Y', '%Y-%m-%d']
    
    for fmt in date_formats:
        try:
            date_obj = datetime.strptime(date_str, fmt)
            return format_date(date_obj.isoformat())
        except ValueError:
            continue

    raise Exception(f"Failed to parse {date_str}")

def standardize_verdict(verdict: Optional[str]) -> Optional[str]:
    """
    Standardize verdict labels
    Maps Politifact verdicts to consistent format
    """
    if not verdict:
        return None
    
    verdict_map = {
        'true': 'true',
        'mostly-true': 'mostly-true',
        'half-true': 'half-true',
        'barely-true': 'mostly-false',
        'mostly-false': 'mostly-false',
        'false': 'false',
        'pants-fire': 'false',
        'pants-on-fire': 'false'
    }
    
    verdict_lower = verdict.lower().strip()
    return verdict_map.get(verdict_lower, verdict_lower).upper()

def clean_politifact_article(article: dict) -> dict:
    """Clean a single Politifact article"""
    cleaned = article.copy()
    
    # Copy fields directly (already at top level)
    cleaned['title'] = clean_text(article.get('title'))
    cleaned['content'] = clean_text(article.get('content'))
    cleaned['claim'] = clean_text(article.get('claim'))
    cleaned['verdict'] = standardize_verdict(article.get('verdict'))
    cleaned['authors'] = article.get('authors', [])
    cleaned['source'] = article.get('source').upper()
    cleaned['type'] = article.get('type').upper() 
    
    # Convert date to ISO string
    cleaned['publish_date'] = parse_date(article['publish_date'])
    
    # Add source bias
    cleaned["source_bias"] = "LEFT-CENTER"

    # Add doc_id
    cleaned["doc_id"] = generate_doc_id(cleaned["url"])
    
    return cleaned

### Load in dataset

In [4]:
with open('../outputs/politifact_synced.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

### Clean 

In [5]:
cleaned_data = [clean_politifact_article(article) for article in tqdm(data, desc="Cleaning articles")]

Cleaning articles:   0%|          | 0/20535 [00:00<?, ?it/s]

### Convert to DF and inspect

In [6]:
df = pd.DataFrame(cleaned_data)

print(f"Total articles: {len(cleaned_data)}")
print(f"\nVerdict distribution:")
print(df['verdict'].value_counts())
print(f"\nSample of cleaned data:")
print(df[['title', 'publish_date', 'verdict']].head())
print(f"\nArticles with missing dates: {df['publish_date'].isna().sum()}")
print(f"Articles with no authors: {df['authors'].apply(len).eq(0).sum()}")

Total articles: 20535

Verdict distribution:
verdict
FALSE           10517
MOSTLY-FALSE     2985
HALF-TRUE        2683
MOSTLY-TRUE      2565
TRUE             1607
FULL-FLOP         118
HALF-FLIP          48
NO-FLIP            12
Name: count, dtype: int64

Sample of cleaned data:
                                               title  \
0  “I have just gotten the highest poll numbers o...   
1  “There’s about 1,400 criminal illegal aliens t...   
2  West Virginia is “the only state losing popula...   
3  A pro-Donald Trump Montana town planned a “ped...   
4  “We’re not cutting science. We’re not cutting ...   

                publish_date verdict  
0  2025-11-24T00:00:00+00:00   FALSE  
1  2025-11-24T00:00:00+00:00   FALSE  
2  2025-11-21T00:00:00+00:00   FALSE  
3  2025-11-21T00:00:00+00:00   FALSE  
4  2025-11-20T00:00:00+00:00   FALSE  

Articles with missing dates: 0
Articles with no authors: 5


### Save cleaned data to final JSON

In [7]:
output_dir = '../outputs_clean/politifact'
os.makedirs(output_dir, exist_ok=True)

with open(f'{output_dir}/politifact_factcheck_cleaned.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_data, f, indent=2, ensure_ascii=False)

print("Cleaning complete! File saved.")

Cleaning complete! File saved.
